# 08 OpenOximetry Raw PPG Window Indices (Phase 4)

Read raw PPG header/dat metadata and attach concrete sample slice indices to each row in the sample window manifest.

Inputs from: `outputs/openox_sample_window_manifest.csv` (from notebook 07).
Outputs to: updated `outputs/openox_sample_window_manifest.csv`, `outputs/openox_pilot_window_summary.csv`, and `outputs/openox_pilot_label_summary.csv`.
Next: notebook 09 for oxygen-reference stream viability audit.

In [ ]:
from pathlib import Path
import re

import numpy as np
import pandas as pd

start = Path.cwd().resolve()
PROJECT_ROOT = None
for candidate in [start, *start.parents]:
    if (candidate / "outputs" / "openox_sample_window_manifest.csv").is_file() and (candidate / "notebooks").is_dir():
        PROJECT_ROOT = candidate
        break

if PROJECT_ROOT is None:
    raise RuntimeError("Could not find project root containing outputs/openox_sample_window_manifest.csv and notebooks/")

OUTPUTS_DIR = PROJECT_ROOT / "outputs"
MANIFEST_CSV = OUTPUTS_DIR / "openox_sample_window_manifest.csv"
WINDOW_SUMMARY_CSV = OUTPUTS_DIR / "openox_pilot_window_summary.csv"
LABEL_SUMMARY_CSV = OUTPUTS_DIR / "openox_pilot_label_summary.csv"

if not MANIFEST_CSV.is_file():
    raise FileNotFoundError(f"Missing manifest CSV: {MANIFEST_CSV}")

print(f"Project root: {PROJECT_ROOT}")
print(f"Manifest CSV: {MANIFEST_CSV}")

## Parse PPG Headers And DAT Layout

Extract record-level metadata from `.hea` and `.dat` files, including sampling rate, total usable samples, channel labels, and start-time anchors for index mapping.

Expected result: manifest rows enriched with validated PPG metadata fields.

In [ ]:
def parse_time_of_day_seconds(value: str | None) -> float | None:
    if value is None:
        return None
    text = str(value).strip()
    if text == "":
        return None
    for fmt in ("%H:%M:%S.%f", "%H:%M:%S"):
        try:
            dt_val = pd.to_datetime(text, format=fmt)
            return dt_val.hour * 3600.0 + dt_val.minute * 60.0 + dt_val.second + dt_val.microsecond / 1_000_000.0
        except ValueError:
            continue
    return None


def signed_circular_day_delta(target_sec: float, start_sec: float) -> float:
    return ((target_sec - start_sec + 43200.0) % 86400.0) - 43200.0


def parse_fmt_bits(fmt_token: str) -> int | None:
    m = re.match(r"^(\d+)", str(fmt_token).strip())
    if not m:
        return None
    return int(m.group(1))


def parse_ppg_record_metadata(hea_path: Path) -> dict[str, object]:
    if not hea_path.is_file():
        return {
            "ppg_metadata_ok": False,
            "ppg_metadata_error": f"Missing .hea file: {hea_path}",
        }

    with hea_path.open("r", encoding="utf-8", errors="ignore") as f:
        lines = [line.rstrip("\n") for line in f if line.strip() != ""]

    if not lines:
        return {
            "ppg_metadata_ok": False,
            "ppg_metadata_error": f"Empty .hea file: {hea_path}",
        }

    first_tokens = lines[0].split()
    if len(first_tokens) < 4:
        return {
            "ppg_metadata_ok": False,
            "ppg_metadata_error": f"Malformed first line in {hea_path}",
        }

    record_name = first_tokens[0]
    channel_count = int(first_tokens[1])
    sampling_rate_hz = float(first_tokens[2])
    total_samples_header = int(first_tokens[3])
    ppg_start_time_text = None
    for tok in first_tokens[4:]:
        if tok.count(":") == 2:
            ppg_start_time_text = tok
            break

    dat_file_token = None
    bytes_per_sample = None
    channel_labels = []
    signal_lines = lines[1 : 1 + channel_count]
    for line in signal_lines:
        toks = line.split()
        if len(toks) >= 2:
            if dat_file_token is None:
                dat_file_token = toks[0]
                bits = parse_fmt_bits(toks[1])
                if bits is not None:
                    bytes_per_sample = int(bits // 8)
        if len(toks) >= 9:
            channel_labels.append(" ".join(toks[8:]))
        else:
            channel_labels.append(None)

    dat_path = hea_path.with_suffix(".dat")
    if dat_file_token is not None:
        candidate = hea_path.parent / dat_file_token
        if candidate.is_file():
            dat_path = candidate

    dat_exists = dat_path.is_file()
    dat_file_size_bytes = int(dat_path.stat().st_size) if dat_exists else None

    bytes_per_frame = None
    total_samples_from_dat = None
    if dat_exists and bytes_per_sample is not None and channel_count > 0:
        bytes_per_frame = int(bytes_per_sample * channel_count)
        if bytes_per_frame > 0:
            total_samples_from_dat = int(dat_file_size_bytes // bytes_per_frame)

    total_samples_used = total_samples_header
    if total_samples_from_dat is not None:
        total_samples_used = min(total_samples_header, total_samples_from_dat)

    return {
        "ppg_metadata_ok": True,
        "ppg_metadata_error": None,
        "record_name": record_name,
        "ppg_channel_count": channel_count,
        "ppg_sampling_rate_hz": sampling_rate_hz,
        "ppg_total_samples_header": total_samples_header,
        "ppg_total_samples_from_dat": total_samples_from_dat,
        "ppg_total_samples_used": total_samples_used,
        "ppg_start_time": ppg_start_time_text,
        "ppg_start_time_seconds": parse_time_of_day_seconds(ppg_start_time_text),
        "ppg_dat_path": str(dat_path),
        "ppg_dat_exists": bool(dat_exists),
        "ppg_dat_file_size_bytes": dat_file_size_bytes,
        "ppg_bytes_per_sample": bytes_per_sample,
        "ppg_bytes_per_frame": bytes_per_frame,
        "ppg_channel_labels": " | ".join([c for c in channel_labels if c]),
    }


manifest_df = pd.read_csv(MANIFEST_CSV)
manifest_df["window_start_timestamp"] = pd.to_datetime(manifest_df["window_start_timestamp"], errors="coerce")
manifest_df["window_end_timestamp"] = pd.to_datetime(manifest_df["window_end_timestamp"], errors="coerce")

unique_hea_paths = sorted({str(p) for p in manifest_df["ppg_hea_path"].dropna().astype(str) if str(p).strip() != ""})
meta_rows = []
for hea_text in unique_hea_paths:
    meta = parse_ppg_record_metadata(Path(hea_text))
    meta["ppg_hea_path"] = hea_text
    meta_rows.append(meta)

meta_df = pd.DataFrame(meta_rows)
manifest_df = manifest_df.merge(meta_df, on="ppg_hea_path", how="left")

print(f"Manifest rows loaded: {len(manifest_df)}")
print(f"Unique PPG records parsed: {len(meta_df)}")

## Compute Concrete Window Slice Indices

Convert manifest window timestamps into raw sample index ranges, clip to valid bounds, and export overlap/in-bounds diagnostics for downstream feature extraction.

In [ ]:
def timestamp_to_seconds_of_day(ts: pd.Timestamp | None) -> float | None:
    if ts is None or pd.isna(ts):
        return None
    return ts.hour * 3600.0 + ts.minute * 60.0 + ts.second + ts.microsecond / 1_000_000.0


start_idx_raw = []
end_idx_raw = []
start_idx_clip = []
end_idx_clip = []
sample_count_clip = []
fully_in_bounds = []
has_overlap = []
slice_offset_bytes = []

for row in manifest_df.itertuples(index=False):
    fs = getattr(row, "ppg_sampling_rate_hz", None)
    total_samples = getattr(row, "ppg_total_samples_used", None)
    start_sec = getattr(row, "ppg_start_time_seconds", None)
    t0 = timestamp_to_seconds_of_day(getattr(row, "window_start_timestamp", None))
    t1 = timestamp_to_seconds_of_day(getattr(row, "window_end_timestamp", None))
    bytes_per_frame = getattr(row, "ppg_bytes_per_frame", None)

    if (
        fs is None
        or pd.isna(fs)
        or total_samples is None
        or pd.isna(total_samples)
        or start_sec is None
        or pd.isna(start_sec)
        or t0 is None
        or t1 is None
    ):
        start_idx_raw.append(np.nan)
        end_idx_raw.append(np.nan)
        start_idx_clip.append(np.nan)
        end_idx_clip.append(np.nan)
        sample_count_clip.append(np.nan)
        fully_in_bounds.append(False)
        has_overlap.append(False)
        slice_offset_bytes.append(np.nan)
        continue

    fs = float(fs)
    total_samples = int(total_samples)
    d0 = signed_circular_day_delta(float(t0), float(start_sec))
    d1 = signed_circular_day_delta(float(t1), float(start_sec))

    i0 = int(round(d0 * fs))
    i1 = int(round(d1 * fs))
    if i1 <= i0:
        i1 = i0 + max(1, int(round((float(t1) - float(t0)) * fs)))

    c0 = max(0, i0)
    c1 = min(total_samples, i1)
    ccount = max(0, c1 - c0)

    start_idx_raw.append(i0)
    end_idx_raw.append(i1)
    start_idx_clip.append(c0)
    end_idx_clip.append(c1)
    sample_count_clip.append(ccount)
    fully_in_bounds.append(i0 >= 0 and i1 <= total_samples)
    has_overlap.append(ccount > 0)
    if bytes_per_frame is None or pd.isna(bytes_per_frame):
        slice_offset_bytes.append(np.nan)
    else:
        slice_offset_bytes.append(int(c0 * int(bytes_per_frame)))

manifest_df["ppg_window_start_index_raw"] = start_idx_raw
manifest_df["ppg_window_end_index_raw"] = end_idx_raw
manifest_df["ppg_window_start_index"] = start_idx_clip
manifest_df["ppg_window_end_index"] = end_idx_clip
manifest_df["ppg_window_sample_count"] = sample_count_clip
manifest_df["ppg_window_fully_in_bounds"] = fully_in_bounds
manifest_df["ppg_window_has_overlap"] = has_overlap
manifest_df["ppg_window_slice_offset_bytes"] = slice_offset_bytes

manifest_df.to_csv(MANIFEST_CSV, index=False)

window_summary_df = pd.DataFrame(
    [
        {"metric": "manifest_rows", "value": int(len(manifest_df))},
        {"metric": "rows_with_ppg_metadata", "value": int(manifest_df["ppg_metadata_ok"].fillna(False).sum())},
        {"metric": "rows_with_index_overlap", "value": int(manifest_df["ppg_window_has_overlap"].fillna(False).sum())},
        {"metric": "rows_fully_in_bounds", "value": int(manifest_df["ppg_window_fully_in_bounds"].fillna(False).sum())},
        {"metric": "mean_window_samples", "value": float(pd.to_numeric(manifest_df["ppg_window_sample_count"], errors="coerce").mean())},
        {"metric": "median_window_samples", "value": float(pd.to_numeric(manifest_df["ppg_window_sample_count"], errors="coerce").median())},
    ]
)
window_summary_df.to_csv(WINDOW_SUMMARY_CSV, index=False)

label_summary_df = (
    manifest_df.groupby("binary_label_so2_lt_90", dropna=False)
    .agg(
        row_count=("encounter_id", "size"),
        encounter_count=("encounter_id", "nunique"),
        mean_so2=("so2_mean", "mean"),
        mean_window_samples=("ppg_window_sample_count", "mean"),
    )
    .reset_index()
)
label_summary_df.to_csv(LABEL_SUMMARY_CSV, index=False)

print(f"Updated manifest: {MANIFEST_CSV}")
print(f"Wrote: {WINDOW_SUMMARY_CSV}")
print(f"Wrote: {LABEL_SUMMARY_CSV}")
window_summary_df